In [1]:
from opt_targeted_transfers import ConditionalTargetedTransfers
from data_loaders import get_dataset
from data_utils import split_data

In [2]:
# Make train and test sets
X, y, r, features = get_dataset("malawi")
d = 3
(X_train, y_train, r_train), (X_test, y_test, r_test) = split_data(X=X[:, :d], y=y, r=r, p=0.6)

In [3]:
# Conditional transfer policy with quantile regression method
cond_tt_qr = ConditionalTargetedTransfers(method="qr", c_bar=2.15, conditional_tolerance=None)

In [4]:
# Since method is "qr," need to set tolerance before fitting.
cond_tt_qr.set_conditional_tolerance(conditional_tolerance=0.1)

In [5]:
# Fit density functions
# Note only training for 50 epochs here for an example, in practice use the default number of epochs.
cond_tt_qr.fit(X_train, y_train, r_train)

Fitting conditional program - QR method via nonparametric regression...


  0%|          | 0/300 [00:00<?, ?it/s, loss=0.0401]

100%|██████████| 300/300 [00:25<00:00, 11.87it/s, loss=0.0135]


In [6]:
# Get optimal policy by solving the optimization problem.
cond_opt_policy = cond_tt_qr.run_opt(
    X_test, r_test
)

In [7]:
# Query the optimal transfer policy after running the optimization algorithm
transfer = cond_opt_policy(X_test[[0]])
print(transfer)

{0: [(1.5520799160003662, 1.0)]}


In [8]:
# Evaluate policy. 
res = cond_tt_qr.evaluate(X_test, y_test, r_test)
res

{'initial_poverty_rate': 0.6511447390812347,
 'initial_poverty_gap': 0.628796805717522,
 'post_transfer_poverty_gap': 0.015505436008921389,
 'post_transfer_poverty_rate': 0.09704073221623183,
 'policy_cost': 1.459357877901334,
 'method': 'conditional_qr',
 'unconditional_tolerance': 0.1,
 'conditional_tolerance': 0.1,
 'd': 3,
 'nclass': None}

In [9]:
# Conditional transfer policy with density estimation method
cond_tt_density = ConditionalTargetedTransfers(method="density", c_bar=2.15, conditional_tolerance=None)

In [10]:
# Fine to fit densities without specifying tolerance if method is "density"
# Note only training for 50 epochs here for an example, in practice use the default number of epochs.
cond_tt_density.fit(X_train, y_train, r_train, n_epochs=50)

KNOTS:[-3.27351677 -3.27351677 -3.27351677 -3.27351677 -1.19700685 -0.84186768
 -0.31661819  0.17797269  5.095039    5.095039    5.095039    5.095039  ]
Fitting conditional densities vs glm spline method...


100%|██████████| 50/50 [01:02<00:00,  1.25s/it, loss=-0.0146, val_loss=0.0177]  

Final Theta: tensor([[-0.9873,  0.7758, -0.2372],
        [ 0.5527, -0.3248, -0.0848],
        [-0.3383,  0.4670, -0.5115],
        [-0.1545, -0.1551,  0.1368],
        [-0.5008,  0.4967,  0.3995],
        [-0.0546, -0.3039,  0.3120],
        [ 0.4300, -0.4868,  0.4648],
        [-0.2917, -0.4789, -0.0600]], dtype=torch.float64)


In [11]:
# Since method is "density," can set tolerance before fitting.
cond_tt_density.set_conditional_tolerance(conditional_tolerance=0.1)

In [12]:
# Get optimal policy by solving the optimization problem.
cond_opt_policy = cond_tt_density.run_opt(
    X_test, r_test
)

In [13]:
# Query the optimal transfer policy after running the optimization algorithm
transfer = cond_opt_policy(X_test[[0]])
print(transfer)

{0: [(1.3704660477280282, 1.0)]}


In [14]:
# Evaluate policy. 
res = cond_tt_density.evaluate(X_test, y_test, r_test)
res

{'initial_poverty_rate': 0.6511447390812347,
 'initial_poverty_gap': 0.628796805717522,
 'post_transfer_poverty_gap': 0.023389805774246403,
 'post_transfer_poverty_rate': 0.11889503294006769,
 'policy_cost': 1.4943111785630634,
 'method': 'conditional_density',
 'unconditional_tolerance': 0.1,
 'conditional_tolerance': 0.1,
 'd': 3,
 'nclass': None}